# Practical No. 06: Retrieval-Augmented Generation (RAG) using a Pre-trained LLM

**Aim:** Build a Retrieval-Augmented Generation (RAG) application that answers questions from a domain-specific document collection, then compare its answers with answers from the same pre-trained LLM without retrieval.

**Domain used:** Artificial Intelligence and Machine Learning (sample knowledge documents).

**Workflow:**
1. Load and prepare domain-specific documents.
2. Split documents into smaller chunks.
3. Convert chunks into embeddings and store them in a vector index.
4. Retrieve the most relevant chunks for a question.
5. Generate an answer using a pre-trained LLM and the retrieved context.
6. Ask the same question without retrieval and compare both answers.

> **Environment:** Designed for Google Colab. Select `Runtime → Change runtime type → T4 GPU` if available. The notebook can also run on CPU, but model loading and generation may be slower.

## 1. Install and import required libraries

We use:
- **Sentence Transformers** to convert text chunks and questions into vector embeddings.
- **FAISS** to find the most similar document chunks.
- **FLAN-T5** as the pre-trained text-generation model.
- **PyPDF** to optionally read PDF documents.

In [1]:
!pip -q install sentence-transformers faiss-cpu transformers accelerate sentencepiece pypdf

In [2]:
import os
import re
import numpy as np
import pandas as pd
import torch
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.1+cpu
CUDA available: False


## 2. Create or load the domain-specific document collection

For a ready-to-run demonstration, the notebook starts with sample documents about AI and Machine Learning. You can replace these with your own domain documents.

To use your own PDFs in Colab, upload them in the optional cell below. Text files can also be added to the `documents` dictionary.

In [3]:
documents = {
    "AI_Introduction": """
    Artificial Intelligence (AI) is a field of computer science that focuses on
    creating systems capable of performing tasks that normally require human
    intelligence. Examples include reasoning, language understanding,
    perception, and decision-making. AI systems may use rules, machine learning,
    or deep learning techniques.
    """,
    "Machine_Learning": """
    Machine Learning (ML) is a subset of artificial intelligence. It enables
    computers to learn patterns from data and make predictions or decisions
    without being explicitly programmed for every task. Common types include
    supervised learning, unsupervised learning, and reinforcement learning.
    """,
    "Supervised_Learning": """
    Supervised learning uses labelled training data. Each training example
    contains input features and a known target output. Classification predicts
    categories, such as spam or not spam. Regression predicts continuous
    values, such as house prices. The model learns a mapping from inputs to
    outputs and is evaluated on unseen data.
    """,
    "Unsupervised_Learning": """
    Unsupervised learning works with data that does not contain labelled
    target outputs. Clustering groups similar data points, while dimensionality
    reduction represents data using fewer features. These methods can help
    discover hidden structure in a dataset.
    """,
    "Deep_Learning": """
    Deep Learning is a branch of machine learning that uses neural networks
    with multiple layers. These networks learn representations from data.
    Deep learning is widely used in image recognition, speech processing,
    natural language processing, and recommendation systems. It often requires
    substantial training data and computing resources.
    """,
    "Natural_Language_Processing": """
    Natural Language Processing (NLP) is an area of AI that enables computers
    to process and generate human language. Common NLP tasks include
    sentiment analysis, text classification, machine translation, summarization,
    and question answering. Modern NLP systems often use transformer models.
    """,
    "Model_Evaluation": """
    A machine learning model should be evaluated using data that was not used
    for training. Accuracy is the proportion of correct predictions in a
    classification task. Precision measures how many predicted positives are
    correct, while recall measures how many actual positives are found.
    The F1-score is the harmonic mean of precision and recall.
    """,
    "Overfitting": """
    Overfitting occurs when a model learns the training data too closely,
    including noise and accidental patterns. It may perform well on training
    data but poorly on new data. Techniques to reduce overfitting include
    regularization, dropout, cross-validation, early stopping, and using more
    representative training data.
    """
}

print("Number of source documents:", len(documents))
for name, text in documents.items():
    print(f"{name}: {len(text.split())} words")

Number of source documents: 8
AI_Introduction: 42 words
Machine_Learning: 39 words
Supervised_Learning: 49 words
Unsupervised_Learning: 35 words
Deep_Learning: 46 words
Natural_Language_Processing: 39 words
Model_Evaluation: 53 words
Overfitting: 44 words


### Optional: Upload PDF documents

Run this cell in Google Colab if you want to add your own PDF files. The extracted text is added to the same document collection. If you do not have PDFs, skip this cell and continue.

In [4]:
# Optional PDF upload cell (Google Colab)
# Uncomment and run when you want to upload PDF files.

# from google.colab import files
# from pypdf import PdfReader
#
# uploaded = files.upload()
# for filename in uploaded.keys():
#     if filename.lower().endswith(".pdf"):
#         reader = PdfReader(filename)
#         pdf_text = "\\n".join(page.extract_text() or "" for page in reader.pages)
#         documents[filename] = pdf_text
#
# print("Total documents after upload:", len(documents))

## 3. Clean and split documents into chunks

Long documents are divided into smaller, overlapping chunks. Chunking helps the retriever find only the relevant part of a document instead of passing an entire document to the LLM.

In [5]:
def clean_text(text):
    text = re.sub(r"\\s+", " ", text)
    return text.strip()

def split_into_chunks(text, chunk_size=100, overlap=20):
    words = clean_text(text).split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        if chunk:
            chunks.append(chunk)
        if end == len(words):
            break
        start = end - overlap

    return chunks

chunk_records = []
for doc_name, doc_text in documents.items():
    for chunk_number, chunk in enumerate(split_into_chunks(doc_text), start=1):
        chunk_records.append({
            "source": doc_name,
            "chunk_number": chunk_number,
            "text": chunk
        })

chunks_df = pd.DataFrame(chunk_records)
print("Total chunks created:", len(chunks_df))
display(chunks_df.head(10))

Total chunks created: 8


,source,chunk_number,text
0,AI_Introduction,1,Artificial Intelligence (AI) is a field of com...
1,Machine_Learning,1,Machine Learning (ML) is a subset of artificia...
2,Supervised_Learning,1,Supervised learning uses labelled training dat...
3,Unsupervised_Learning,1,Unsupervised learning works with data that doe...
4,Deep_Learning,1,Deep Learning is a branch of machine learning ...
5,Natural_Language_Processing,1,Natural Language Processing (NLP) is an area o...
6,Model_Evaluation,1,A machine learning model should be evaluated u...
7,Overfitting,1,Overfitting occurs when a model learns the tra...


## 4. Generate embeddings and create a FAISS vector index

An embedding is a numerical representation of text. Texts with similar meanings tend to have nearby vectors. FAISS searches these vectors efficiently to retrieve relevant chunks.

We use the pre-trained `all-MiniLM-L6-v2` sentence-transformer model for embeddings.

In [6]:
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_name)

chunk_texts = chunks_df["text"].tolist()
embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

# Normalize vectors so inner-product search is equivalent to cosine similarity.
faiss.normalize_L2(embeddings)

vector_dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(vector_dimension)
index.add(embeddings)

print("Embedding shape:", embeddings.shape)
print("Vectors stored in FAISS:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (8, 384)
Vectors stored in FAISS: 8


## 5. Define the document retrieval function

The question is converted into an embedding. FAISS then returns the top-k chunks with the highest cosine similarity scores.

In [7]:
def retrieve_documents(question, top_k=3):
    question_embedding = embedding_model.encode(
        [question], convert_to_numpy=True
    ).astype("float32")
    faiss.normalize_L2(question_embedding)

    scores, indices = index.search(question_embedding, min(top_k, len(chunk_texts)))

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        row = chunks_df.iloc[int(idx)]
        results.append({
            "source": row["source"],
            "chunk_number": int(row["chunk_number"]),
            "score": float(score),
            "text": row["text"]
        })
    return results

# Test retrieval
test_question = "What is supervised learning?"
retrieved = retrieve_documents(test_question, top_k=3)

for number, item in enumerate(retrieved, start=1):
    print(f"\nResult {number} | Source: {item['source']} | Similarity: {item['score']:.3f}")
    print(item["text"])


Result 1 | Source: Supervised_Learning | Similarity: 0.697
Supervised learning uses labelled training data. Each training example contains input features and a known target output. Classification predicts categories, such as spam or not spam. Regression predicts continuous values, such as house prices. The model learns a mapping from inputs to outputs and is evaluated on unseen data.

Result 2 | Source: Machine_Learning | Similarity: 0.556
Machine Learning (ML) is a subset of artificial intelligence. It enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task. Common types include supervised learning, unsupervised learning, and reinforcement learning.

Result 3 | Source: Unsupervised_Learning | Similarity: 0.483
Unsupervised learning works with data that does not contain labelled target outputs. Clustering groups similar data points, while dimensionality reduction represents data using fewer features. These meth

## 6. Load the pre-trained LLM

We use Google's pre-trained **FLAN-T5-small** instruction-following model. It is relatively lightweight and can run without an API key. The same model is used for both RAG and the no-retrieval baseline so that the comparison focuses on the effect of retrieval.

In [9]:
!pip install -q transformers==4.48.3

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [10]:
llm_model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(llm_model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("LLM loaded:", llm_model_name)
print("Generation device:", device)

print("LLM loaded:", llm_model_name)
print("Generation device:", "GPU" if device == 0 else "CPU")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM loaded: google/flan-t5-small
Generation device: cpu
LLM loaded: google/flan-t5-small
Generation device: CPU


## 7. Build the RAG answer-generation function

The RAG prompt contains the user's question and the retrieved document chunks. The model is instructed to answer using the supplied context and to say when the context does not contain the answer.

In [12]:

def generate_text(prompt, max_new_tokens=160):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


def answer_with_rag(question, top_k=3):
    retrieved = retrieve_documents(question, top_k=top_k)

    context_parts = []
    for item in retrieved:
        context_parts.append(
            f"[Source: {item['source']}] {item['text']}"
        )

    context = "\n".join(context_parts)

    prompt = f"""Answer the question using only the context provided below.
If the context does not contain the answer, say:
"The provided documents do not contain enough information."

Context:
{context}

Question: {question}
Answer:
"""

    answer = generate_text(prompt)

    return answer, retrieved


# Test the RAG application
sample_question = "What is supervised learning and what are its types of tasks?"

rag_answer, rag_sources = answer_with_rag(sample_question)

print("QUESTION:", sample_question)
print("\nRAG ANSWER:", rag_answer)

print("\nRETRIEVED SOURCES:")
for source in rag_sources:
    print(f"- {source['source']} (similarity: {source['score']:.3f})")


QUESTION: What is supervised learning and what are its types of tasks?

RAG ANSWER: a subset of artificial intelligence

RETRIEVED SOURCES:
- Supervised_Learning (similarity: 0.651)
- Machine_Learning (similarity: 0.548)
- Model_Evaluation (similarity: 0.432)


## 8. Generate an answer without retrieval (LLM-only)

For a fair comparison, we use the same FLAN-T5 model but do not provide any retrieved documents. The model answers from its pre-trained knowledge.

In [14]:

def answer_without_rag(question):
    prompt = f"""
Answer the following question clearly and concisely.

Question: {question}
Answer:
"""
    return generate_text(prompt)


# Generate answer without retrieval
llm_only_answer = answer_without_rag(sample_question)

print("QUESTION:", sample_question)
print("\nLLM-ONLY ANSWER:", llm_only_answer)


QUESTION: What is supervised learning and what are its types of tasks?

LLM-ONLY ANSWER: supervised learning


## 9. Compare RAG and LLM-only responses

Run the same questions through both approaches. Review whether the RAG answer uses information from the supplied documents and whether the LLM-only answer introduces unsupported or missing details.

The comparison below is a practical, qualitative comparison. It is not an automatic factuality score.

In [15]:
questions = [
    "What is supervised learning?",
    "What is the difference between supervised and unsupervised learning?",
    "What is overfitting and how can it be reduced?",
    "What is Natural Language Processing?",
    "Which metrics are used to evaluate a classification model?",
    "What is the capital of Japan?"
]

comparison_rows = []

for question in questions:
    rag_response, sources = answer_with_rag(question, top_k=3)
    plain_response = answer_without_rag(question)

    comparison_rows.append({
        "Question": question,
        "RAG Answer": rag_response,
        "LLM-only Answer": plain_response,
        "Retrieved Sources": ", ".join(dict.fromkeys(s["source"] for s in sources))
    })

comparison_df = pd.DataFrame(comparison_rows)
pd.set_option("display.max_colwidth", 250)
display(comparison_df)

,Question,RAG Answer,LLM-only Answer,Retrieved Sources
0,What is supervised learning?,a subset of artificial intelligence,supervised learning,"Supervised_Learning, Machine_Learning, Unsupervised_Learning"
1,What is the difference between supervised and unsupervised learning?,Unsupervised learning works with data that does not contain labelled target outputs,supervised learning is a learning process that involves learning to learn.,"Unsupervised_Learning, Supervised_Learning, Machine_Learning"
2,What is overfitting and how can it be reduced?,"Regularization, dropout, cross-validation, early stopping, and using more representative training data",a slender slender slender slender,"Overfitting, Model_Evaluation, Supervised_Learning"
3,What is Natural Language Processing?,an area of AI that enables computers to process and generate human language,Natural language processing,"Natural_Language_Processing, AI_Introduction, Supervised_Learning"
4,Which metrics are used to evaluate a classification model?,Precision,arithmetic,"Model_Evaluation, Supervised_Learning, Unsupervised_Learning"
5,What is the capital of Japan?,Japan,Tokyo,"AI_Introduction, Model_Evaluation, Supervised_Learning"


## 10. Inspect retrieval results for a question

This cell displays the retrieved chunks and similarity scores. A higher cosine similarity score indicates that the chunk is more similar to the question in embedding space; it does not guarantee that the chunk fully answers the question.

In [16]:
inspection_question = "How can overfitting be reduced?"
inspection_results = retrieve_documents(inspection_question, top_k=4)

inspection_df = pd.DataFrame([
    {
        "Rank": i + 1,
        "Source": item["source"],
        "Similarity Score": round(item["score"], 3),
        "Retrieved Text": item["text"]
    }
    for i, item in enumerate(inspection_results)
])

display(inspection_df)

,Rank,Source,Similarity Score,Retrieved Text
0,1,Overfitting,0.767,"Overfitting occurs when a model learns the training data too closely, including noise and accidental patterns. It may perform well on training data but poorly on new data. Techniques to reduce overfitting include regularization, dropout, cross-va..."
1,2,Unsupervised_Learning,0.190,"Unsupervised learning works with data that does not contain labelled target outputs. Clustering groups similar data points, while dimensionality reduction represents data using fewer features. These methods can help discover hidden structure in a..."
2,3,Model_Evaluation,0.187,"A machine learning model should be evaluated using data that was not used for training. Accuracy is the proportion of correct predictions in a classification task. Precision measures how many predicted positives are correct, while recall measures..."
3,4,Supervised_Learning,0.186,"Supervised learning uses labelled training data. Each training example contains input features and a known target output. Classification predicts categories, such as spam or not spam. Regression predicts continuous values, such as house prices. T..."


## 11. Observations and result

After executing the notebook, record observations based on the generated outputs.

**Expected observations:**
1. The retriever searches the document collection and selects chunks relevant to the question.
2. RAG gives the LLM additional domain-specific context from the supplied documents.
3. The LLM-only approach does not access the document collection and may provide a general answer or miss document-specific details.
4. RAG answers can be easier to verify because retrieved source names are displayed.
5. RAG quality depends on document quality, chunk size, embedding quality, retrieval settings, and the LLM.
6. RAG does not guarantee correctness. Irrelevant retrieval or incomplete documents can still lead to incorrect answers.

**Result:** A RAG application was implemented using a pre-trained sentence-transformer embedding model, FAISS similarity search, and the FLAN-T5-small pre-trained LLM. Its responses were compared with responses generated by the same LLM without retrieval.

## 12. Conclusion

The practical demonstrates how Retrieval-Augmented Generation combines information retrieval with text generation. The document collection is split into chunks and converted into embeddings. FAISS retrieves relevant chunks for a user query, and the pre-trained LLM generates an answer using that context. In comparison, the LLM-only system generates a response without access to the supplied documents. RAG is useful when answers should be grounded in a specific collection of documents, while the quality of the result depends on the relevance and accuracy of the retrieved context.
